# C-MET Colab Demo Reproduction

This notebook runs the official C-MET inference demo for **Cross-Modal Emotion Transfer for Emotion Editing in Talking Face Video**.

Use a CUDA runtime first: `A100 GPU > L4 GPU > T4 GPU`. The official `inference.py` contains direct `.cuda()` calls, so CPU/Mac/MPS is not the right first target.

Goal: generate one basic emotion video and one extended emotion video before trying any training.

## 1. Check GPU

Before doing anything expensive, confirm Colab actually gave you a GPU.

In [ ]:
!nvidia-smi
!python --version

## 2. Clone Official C-MET

This reproduction repo is intentionally lightweight. It does not vendor the official C-MET code or checkpoints.

In [ ]:
!rm -rf C-MET
!git clone https://github.com/ChanHyeok-Choi/C-MET.git
%cd C-MET

## 3. Install System Dependencies

The official README recommends `ffmpeg=4.4` through conda. In Colab, apt ffmpeg is usually enough for the first demo.

In [ ]:
!apt-get update -y
!apt-get install -y ffmpeg pkg-config
!ffmpeg -version | head -n 3

## 4. Install Minimal Python Dependencies

The official `requirements.txt` is heavy and may fail on Colab's Python 3.12 because it includes training, TTS, ASR, Gradio, and super-resolution packages. For the first demo, install only the inference dependencies and run without `--sr`.

In [ ]:
!python -m pip install --upgrade pip setuptools wheel
!python -m pip install --upgrade --no-cache-dir \
  huggingface_hub omegaconf tqdm pandas scipy \
  librosa==0.10.2.post1 'soundfile>=0.12.1' \
  moviepy==1.0.3 imageio-ffmpeg av opencv-python-headless pillow

## 4.1 Patch Optional Imports And Compatibility

`funasr` is imported by the official script but is not used when we provide pre-extracted `emotion2vec+large` features. The audio utility can also hit old `librosa`/NumPy alias issues on Colab. `moviepy.editor` can also fail on Python 3.12 through an old pygame/pkg_resources path. These small patches keep the demo path lightweight.

In [ ]:
from pathlib import Path

path = Path("inference.py")
text = path.read_text()
text = text.replace(
    "from funasr import AutoModel\n",
    "try:\n    from funasr import AutoModel\nexcept Exception:\n    AutoModel = None\n",
)
path.write_text(text)

audio_path = Path("src/audio.py")
audio_text = audio_path.read_text()
audio_text = audio_text.replace(
    "import librosa\nimport librosa.filters\nimport numpy as np\n",
    "import numpy as np\n"
    "if not hasattr(np, 'complex'):\n    np.complex = complex\n"
    "if not hasattr(np, 'float'):\n    np.float = float\n"
    "if not hasattr(np, 'int'):\n    np.int = int\n"
    "import librosa\nimport librosa.filters\n",
)
audio_path.write_text(audio_text)

for moviepy_path in [Path("inference.py"), Path("src/util.py")]:
    moviepy_text = moviepy_path.read_text()
    moviepy_text = moviepy_text.replace(
        "from moviepy.editor import *\n",
        "# moviepy.editor is only needed for optional --sr post-processing.\n"
        "VideoFileClip = None\n"
        "AudioFileClip = None\n",
    )
    moviepy_path.write_text(moviepy_text)

print("Patched optional funasr import, librosa/NumPy compatibility, and moviepy import")

## 4.2 Verify Audio Dependencies

This catches the common `np.complex`/`librosa` issue before the long inference step.

In [ ]:
import numpy as np
import librosa
print("numpy:", np.__version__)
print("librosa:", librosa.__version__)

## 5. Verify PyTorch CUDA

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 6. Run Basic Emotion Demo: Happy

The first run may download C-MET and EDTalk weights from Hugging Face. Do not add `--sr` yet.

In [ ]:
!python inference.py --num_samples 10 \
  --connector_exp_path ./checkpoints/_epoch_2105_checkpoint_step000200000.pth \
  --source_path ./asset/identity/ChatGPT_man3_crop.png \
  --audio_driving_path ./asset/audio/W009_038.wav \
  --pose_driving_path ./asset/video/W009_038.mp4 \
  --save_path ./res/ChatGPT_man3_happy.mp4 \
  --neu_e2v_path ./audios/MEAD/neutral/emotion2vec+large_features/ \
  --emo_e2v_path ./audios/MEAD/happy/emotion2vec+large_features/

## 7. Display Happy Result

In [ ]:
from IPython.display import Video, display
display(Video("./res/ChatGPT_man3_happy.mp4", embed=True))

## 8. Run Extended Emotion Demo: Sarcastic

This is the more report-friendly example because the paper emphasizes extended emotions.

In [ ]:
!python inference.py --num_samples 10 \
  --connector_exp_path ./checkpoints/_epoch_2105_checkpoint_step000200000.pth \
  --source_path ./asset/identity/ChatGPT_man3_crop.png \
  --audio_driving_path ./asset/audio/W009_038.wav \
  --pose_driving_path ./asset/video/W009_038.mp4 \
  --save_path ./res/ChatGPT_man3_sarcastic.mp4 \
  --neu_e2v_path ./audios/MEAD/neutral/emotion2vec+large_features/ \
  --emo_e2v_path ./audios/gemini/sarcastic/emotion2vec+large_features/

## 9. Display Sarcastic Result

In [ ]:
display(Video("./res/ChatGPT_man3_sarcastic.mp4", embed=True))

## 10. Save Outputs To Drive

Optional, but useful before the Colab runtime disconnects.

In [ ]:
# Optional: uncomment if you want to persist results in Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/C-MET-results
# !cp ./res/*.mp4 /content/drive/MyDrive/C-MET-results/

## What Counts As Success

- `./res/ChatGPT_man3_happy.mp4` exists and plays.
- `./res/ChatGPT_man3_sarcastic.mp4` exists and plays.
- You recorded GPU type, command, and any errors in your experiment log.

After this works, the next reproduction step is small-sample training, not full MEAD training.